# Sparse-PSAM RNN on the *controlled* (composition-deconfounded) oracle

This is the `rnn-psam-display` experiment, but the training **label** is swapped.

- `rnn-psam-display` trains `r_rnn_500_1l_sparse_10_psams` to predict **raw SpliceAI**'s exon call.
- Here we train the *same* architecture and *same* sparse-gate schedule against the **controlled oracle**: SpliceAI's exon score with a monotonic bag-of-k-mers composition gate subtracted and median-thresholded (`gate_composition_residual.gate_residual_oracle`).

**Question.** With composition regressed out, where does a sparse model put its 10 PSAM kernels — on splice-site / positional structure, on frame / stop-codon structure, or on nothing coherent?

Only the label source differs; the sparse gate, RNN, BCE-in-bits loss, epoch loop and adaptive-sparsity schedule are reused unchanged (`orthogonal_dfa.experiments.controlled_oracle_rnn`, which mirrors `train_rnn_psams_sparse`).

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import matplotlib.pyplot as plt
import numpy as np
from render_psam import render_psams

from orthogonal_dfa.experiments.controlled_oracle_rnn import (
    train_sparse_psams_controlled,
    final_sparse_gate,
)
from orthogonal_dfa.experiments.sparse_ngrams import compute_sparse_results, get_all_logos

## Configuration

Defaults match `r_rnn_500_1l_sparse_10_psams` (500 units, 1 layer, 10 PSAMs, `initial_threshold=0.60`, 20k epochs, 100k samples/epoch, fresh data every 5 epochs, 50 finetune epochs).

The full 20k-epoch run is long; training is permacached in 500-epoch chunks (resumable), and the controlled-oracle labels are permacached too. For a quick smoke test, set `EPOCHS` small (e.g. `500`). The registered SpliceAI model trains 10 seeds and keeps the best; here we run one seed by default (change `SEED` / loop for more).

In [ ]:
SEED = 0
EPOCHS = 20_000  # set to e.g. 500 for a quick smoke test
NUM_PSAMS = 10

## Train the sparse-PSAM RNN on the controlled oracle

Building the controlled oracle fits the composition gate on first use (cached per process); labelling 100k middles/epoch runs SpliceAI, permacached by `(count, seed)`.

In [ ]:
gate_trained, results = train_sparse_psams_controlled(
    seed=SEED, epochs=EPOCHS, num_psams=NUM_PSAMS
)
gate = final_sparse_gate(results, gate_trained)
print("final sparse gate selected from", len(results), "epochs of results")

## Learned sparse PSAM kernels, as logos

`gate.phi.psams.sequence_logos` are the 10 learned convolutional kernels (width 9) as log-odds logos — the model's motifs for the controlled signal. Rendered `psam_mode="raw"` (raw log-odds), as in `rnn-psam-display`.

In [ ]:
p = gate.phi.psams
render_psams(
    p.sequence_logos,
    names=[f"PSAM {i}" for i in range(NUM_PSAMS)],
    psam_mode="raw",
)
plt.show()

## Empirical logos from the sparse activations

For each PSAM, gather the 9-mers under its non-zero sparse activations across a fresh sample of middles and average them — the sequences the kernel actually fires on. Rendered `psam_mode="info"` (information-content sequence logos).

`compute_sparse_results` draws its middles from `sample_text` (oracle-independent — it discards the labels), so it applies unchanged to the controlled-trained model.

In [ ]:
phi = gate.phi.eval()
x, m = compute_sparse_results(phi, seed=0)
logos_empirical = get_all_logos(x, m)
render_psams(
    logos_empirical,
    names=[f"PSAM {i}" for i in range(NUM_PSAMS)],
    psam_mode="info",
)
plt.show()

## What to look for

- **Splice-site / positional motifs** (donor `GT…`/acceptor `…AG` consensus, branch-point) would say the deconfounded signal is the splice-site geometry.
- **Stop-codon fragments** (`TAA`/`TAG`/`TGA`) across the kernels would say the model keys on reading-frame closure.
- **Low-information / incoherent kernels** would say that, with composition removed, there is little compact motif structure left for a sparse model to grab — the RNN counterpart of the direct-L\* result.

Compare against `rnn-psam-display.ipynb` (same architecture, raw-SpliceAI label) to see which motifs survive composition-deconfounding.